# Week 12 — Guided project

**Notebook outcomes**

- Stitch the camp's tools together into one end-to-end project
- Produce a small, reproducible research artifact
- Reflect on what went well and where tools earned their keep


## The project

You'll build a small package that simulates and analyzes a
**two-state Markov chain** — the simplest possible dynamic model. It's
mathematically tiny, but it gives you a real excuse to exercise every
tool in the camp:

- **uv** for the environment
- **Git** for version control
- **Python** for the implementation
- **pytest** for tests
- **LaTeX** for a short writeup
- **slidev** (optional stretch) for a short deck

Economically: you can imagine the two states as `employed` /
`unemployed`, `boom` / `recession`, `rich` / `poor`, etc. The package
should be agnostic to the labels.


## The math

A two-state Markov chain is specified by a 2×2 transition matrix
$$
P = \begin{pmatrix} p_{00} & p_{01} \\ p_{10} & p_{11} \end{pmatrix}
$$
with rows summing to 1. Each row is the distribution over "where you go
next" conditional on your current state.

We want three quantities:

1. **A simulated path** of length $T$ — a sequence of 0/1 values.
2. **The empirical frequency** of each state in that path.
3. **The stationary distribution** $\pi$ satisfying $\pi P = \pi$.

For two states, the stationary distribution has a closed form:
$$
\pi_0 = \frac{p_{10}}{p_{01} + p_{10}}, \qquad
\pi_1 = \frac{p_{01}}{p_{01} + p_{10}}.
$$

Check: as $T \to \infty$ the empirical frequency should converge to
$\pi$.


## Required deliverables

A Git repository containing:

```
camp_project/
├── README.md
├── pyproject.toml
├── uv.lock
├── src/
│   └── camp_project/
│       ├── __init__.py
│       ├── markov.py          # simulation + stationary distribution
│       └── cli.py             # (stretch) command-line entry point
├── tests/
│   └── test_markov.py
├── notebooks/
│   └── analysis.ipynb         # uses your package to produce plots
└── paper/
    ├── main.tex               # ~1 page writeup
    └── refs.bib
```

### Must-haves

- `markov.py` has:
  - `simulate(P: list[list[float]], T: int, seed: int = 0) -> list[int]`
  - `empirical_frequency(path: list[int]) -> dict[int, float]`
  - `stationary_distribution(P: list[list[float]]) -> dict[int, float]`
- Each function has a docstring and type hints.
- `tests/test_markov.py` has **at least 6** tests:
  - rows summing to 1 is enforced (or at least checked)
  - stationary distribution on a symmetric `P` is `{0: 0.5, 1: 0.5}`
  - `simulate` with a degenerate `P` (e.g., `[[1,0],[0,1]]`) returns a
    constant path
  - `simulate` produces paths of the expected length
  - `empirical_frequency` sums to 1
  - a known-answer case you design

- `analysis.ipynb` uses your package to:
  - Pick a `P` (not symmetric).
  - Simulate one long path.
  - Plot the running empirical frequency converging to the stationary
    distribution (matplotlib is fine).

- `paper/main.tex` is ~1 page and includes:
  - An abstract/intro
  - The math (Bellman-flavored explanation of what "stationary"
    means)
  - One plot from your notebook (`\includegraphics`)
  - One citation (any source — Markov chain textbook, Wikipedia,
    a paper)

- `ruff check .` passes.
- `pytest` passes.


### Stretch goals

If you finish the above and want more:

- **CLI.** Add an entry point in `cli.py` + `pyproject.toml` so
  `uv run camp-project --T 10000` prints the empirical and stationary
  distributions.
- **Generalization.** Extend to an arbitrary $k$-state chain. Keep the
  API the same. The closed form for stationary disappears; solve the
  linear system numerically.
- **Slidev.** A 5-slide deck summarizing your findings.
- **CI.** Add a tiny GitHub Actions workflow that runs `pytest` on push.


## Suggested workflow

1. Make the repo. Commit the scaffold.
2. Write `simulate` with a small test. Commit.
3. Write `empirical_frequency` with a small test. Commit.
4. Write `stationary_distribution` with a known-answer test. Commit.
5. Refactor / lint / add docstrings. Commit.
6. Open a branch called `writeup`. Add the LaTeX skeleton. Commit.
7. In `analysis.ipynb`, generate the figure. Save it as
   `paper/figures/convergence.pdf`. Commit.
8. Flesh out `main.tex`. Reference the figure. Commit.
9. Open a PR (against `main`), review your own diff, merge.
10. Push tag `v1.0`. You're done.

Don't do it in one sitting. The repo's commit history should look like
you worked iteratively — because you did.


## What we're grading

Priorities, roughly:

1. **It runs.** `uv sync`, `uv run pytest`, `uv run jupyter lab`,
   `tectonic paper/main.tex` should all work on a fresh clone.
2. **It's correct.** Stationary distribution matches empirical
   frequency on a long path. Math in the writeup is right.
3. **It's readable.** Good names, docstrings, small functions.
4. **Stylish tooling.** Ruff clean, useful commit messages, sensible
   `.gitignore`.

Not graded: how pretty the plot is, whether you picked an impressive
transition matrix.


## Example scaffold (copy and adapt)


In [ ]:
# src/camp_project/markov.py (sketch — do not import from the notebook; write it in your package)
import random


def simulate(P, T, seed=0):
    """Simulate T steps of a Markov chain with transition matrix P.

    Start in state 0. Return a list of integers of length T.
    """
    rng = random.Random(seed)
    state = 0
    out = [state]
    for _ in range(T - 1):
        r = rng.random()
        probs = P[state]
        cumulative = 0.0
        for next_state, p in enumerate(probs):
            cumulative += p
            if r < cumulative:
                state = next_state
                break
        out.append(state)
    return out


def empirical_frequency(path):
    """Return a dict mapping state to its empirical frequency in `path`."""
    counts = {}
    for s in path:
        counts[s] = counts.get(s, 0) + 1
    n = len(path)
    return {s: c / n for s, c in counts.items()}


def stationary_distribution(P):
    """Closed-form stationary distribution for a 2-state chain."""
    p01 = P[0][1]
    p10 = P[1][0]
    pi0 = p10 / (p01 + p10)
    return {0: pi0, 1: 1 - pi0}

## Tips from instructors

- **Commit early, commit small.** If your commit history has one commit
  "done", we'll notice.
- **Write the first test before the second function.** It's faster than
  you think, and it catches API problems early.
- **Use AI assistance** for boilerplate (docstrings, LaTeX skeletons).
  Don't use it to do the math for you.
- **Ask questions.** Office hours and the camp Slack are there.


## Recap

This is where the previous 11 weeks pay off. Every tool has a role. No
single thing is hard; the practice is *integrating* them.

Good luck!
